In [ ]:
import numpy as np
from copy import copy
import cmath
import math

from matplotlib import pyplot as plt

from numpy import linalg as LA
from scipy.linalg import null_space
from scipy.optimize import fsolve
from scipy.optimize import root
from scipy.optimize import newton_krylov

import pickle
import pandas as pd

In [ ]:
%matplotlib notebook

In [ ]:
# Pauli matrices

sigma_x = np.array([[0, 1], [1, 0]])
sigma_y = np.array([[0, -1j], [1j, 0]])

In [ ]:
phi_0 = np.linspace(0, 2*np.pi, num=201)

# length
L = 1.5 

# magnetic field
b_x = 0.0 
b_y = 0.0
b_z = 0.2

# SO coupling, alpha >= 0
gamma_SO = 0.14 

# k_F = v_F * m = m
k_F = 0.1

In [ ]:
tau_1 = 0.75
tau_2 = 0.75

r_1 = np.sqrt(1 - tau_1)
r_2 = np.sqrt(1 - tau_2)

In [ ]:
lambda_coupl = 0.1 # Andreev-continuum coupling rate
damping_param = 0.01 
Omega = 0.01

T_bos = 0.07

alpha_0 = 0.0001
omega_c = 1

In [ ]:
def bose_distrib(x, T):
    
    if x>=0:
        n_B = 1/(np.exp(x/T)-1)
        return n_B
    else:
        n_B = 1/(np.exp(-x/T)-1)
        return -1-n_B

In [ ]:
# spectral density function

def spectral_density(x, lambda_coupl, damping_param, Omega):
    
    J = lambda_coupl**2 * damping_param * (1/((x - Omega)**2 + damping_param**2/4) - \
                                                   1/((x + Omega)**2 + damping_param**2/4))
    
    J_ohm = alpha_0 * x * np.exp(-np.abs(x)/omega_c)
    
    return J + J_ohm

Transfer matrix functions

In [ ]:
A_pm = lambda E, s: np.array([[ 1j * s * (E - (b_z + s*gamma_SO*k_F)) / (1 + s*gamma_SO), \
                               -1j * s * (b_x - 1j*b_y) / (1 + s*gamma_SO) ], \
                             [ -1j * s * (b_x + 1j*b_y) / (1 - s*gamma_SO), \
                                1j * s * (E + (b_z + s*gamma_SO*k_F)) / (1 - s*gamma_SO) ]])

In [ ]:
# spin TM without assumption of small SOI constant

def spin_transf(E):
    
    A_plus = np.array(A_pm(E, 1), dtype='complex')
    A_minus = np.array(A_pm(E, -1), dtype='complex')
    
    eigenval_pl, eigenvect_pl = LA.eig(A_plus)
    eigenval_min, eigenvect_min = LA.eig(A_minus)
    
    W_pl = eigenvect_pl @ np.array([[np.exp(-eigenval_pl[0]*L), 0], [0, np.exp(-eigenval_pl[1]*L)]]) @ \
         LA.inv(eigenvect_pl)
    
    W_min = eigenvect_min @ np.array([[np.exp(-eigenval_min[0]*L), 0], [0, np.exp(-eigenval_min[1]*L)]]) @ \
         LA.inv(eigenvect_min)
    
    return W_pl, W_min

In [ ]:
def TM_ABS(E, i):
    
    # particles
    W_pl_n, W_min_n = spin_transf(E)
    
    TM = np.exp(1j * phi_0[i] / 2) * \
                np.block([[ (E + 1j*cmath.sqrt(1-E**2)) * (W_pl_n + r_1*r_2*W_min_n),\
                r_2*W_pl_n + r_1*W_min_n],\
                [r_1*W_pl_n + r_2*W_min_n, \
                (E - 1j*cmath.sqrt(1-E**2)) * (W_min_n + r_1*r_2*W_pl_n)]])
    
    return TM

In [ ]:
def TM_ABS_Nambu(E, i):
    
    # holes
    W_pl_nn, W_min_nn = spin_transf(-E)
    W_pl_nh = sigma_y @ np.conj(W_pl_nn) @ sigma_y
    W_min_nh = sigma_y @ np.conj(W_min_nn) @ sigma_y
    
    TM_N = np.exp(-1j * phi_0[i] / 2) * \
                np.block([[ (E - 1j*cmath.sqrt(1-E**2)) * (W_min_nh + r_1*r_2*W_pl_nh),\
                r_2*W_min_nh + r_1*W_pl_nh ],\
                [r_1*W_min_nh + r_2*W_pl_nh , \
                (E + 1j*cmath.sqrt(1-E**2)) * (W_pl_nh + r_1*r_2*W_min_nh) ]])
    
    return TM_N

In [ ]:
def TM_cont(E, i):
    
    # particles
    W_pl_n, W_min_n = spin_transf(E)
    
    TM_c = np.exp(1j * phi_0[i] / 2) * \
                np.block([[ (np.abs(E) + cmath.sqrt(E**2-1)) * (W_pl_n + r_1*r_2*W_min_n),\
                r_2*W_pl_n + r_1*W_min_n],\
                [r_1*W_pl_n + r_2*W_min_n, \
                (np.abs(E) - cmath.sqrt(E**2-1)) * (W_min_n + r_1*r_2*W_pl_n)]])
    
    return TM_c

In [ ]:
def TM_cont_Nambu(E, i):
    
    # holes
    W_pl_nn, W_min_nn = spin_transf(-E)
    W_pl_nh = sigma_y @ np.conj(W_pl_nn) @ sigma_y
    W_min_nh = sigma_y @ np.conj(W_min_nn) @ sigma_y
    
    TM_N_c = np.exp(-1j * phi_0[i] / 2) * \
                np.block([[ (np.abs(E) - cmath.sqrt(E**2-1)) * (W_min_nh + r_1*r_2*W_pl_nh),\
                r_2*W_min_nh + r_1*W_pl_nh ],\
                [r_1*W_min_nh + r_2*W_pl_nh , \
                (np.abs(E) + cmath.sqrt(E**2-1)) * (W_pl_nh + r_1*r_2*W_min_nh) ]])
    
    return TM_N_c

In [ ]:
def det_Matr_new(E, i):
      
    M_old =  TM_ABS(E, i) - TM_ABS_Nambu(E, i)

    return LA.det(M_old)

In [ ]:
# gather all roots in a single vector!
# 4 for spinless, 8 for spinfull case

roots = np.zeros((len(phi_0), 8))

for i in range(len(phi_0)):
  
    roots[i, 0] = fsolve(det_Matr_new, 0.3, args=(i))  
    roots[i, 1] = fsolve(det_Matr_new, 0.6, args=(i))    
    roots[i, 2] = fsolve(det_Matr_new, 0.78, args=(i))   
    roots[i, 3] = fsolve(det_Matr_new, 0.98, args=(i))    
    
    roots[i, 4] = fsolve(det_Matr_new, -0.1, args=(i))
    roots[i, 5] = fsolve(det_Matr_new, -0.6, args=(i))     
    roots[i, 6] = fsolve(det_Matr_new, -0.78, args=(i))
    roots[i, 7] = fsolve(det_Matr_new, -0.98, args=(i))

In [ ]:
plt.plot(phi_0/np.pi, roots[:,0], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,1], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,2], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,3], linewidth=1.5)


plt.grid()
plt.title(r"$\tau_1={}, \tau_2={}, k_F = {}, \gamma_{{SO}} = {},$"\
          .format(tau_1, tau_2, k_F, gamma_SO)+'\n'+
          r"$b_x={}\Delta, b_y = {}\Delta, b_z = {}\Delta, L={}\Delta^{{-1}}$"\
          .format(np.round(b_x, 2), b_y, np.round(b_z, 2), L), fontsize=13)
plt.xlabel(r'$\varphi$, $\pi$', fontsize=12)
plt.ylabel(r'$E_A$, $\Delta$', fontsize=12)


### Finding the nullspace of M - c, d amplitudes (for each phi_0)

In [ ]:
def Matr(E, i):
    
    
    M_old = (TM_ABS(E, i) - TM_ABS_Nambu(E, i)) / np.sqrt(tau_1*tau_2) 
    
    M_up = np.array([[M_old[0, 0], M_old[0, 2]], [M_old[2, 0], M_old[2, 2]]])
    M_down = np.array([[M_old[1, 1], M_old[1, 3]], [M_old[3, 1], M_old[3, 3]]])
    
    return M_old

In [ ]:
def ab_vect(cd_vect, E, i):
    
    M_old =  1 / 2 / np.sqrt(tau_1*tau_2) * (TM_ABS(E, i) + TM_ABS_Nambu(E, i))
    ab = M_old @ cd_vect
    
    return ab

In [ ]:
sol_n = 8

# cd and abb arrays for each phi_0[i], for each band, containing 4 values:
#(c_up, c_down, d_up, d_down)       (a_up, a_down, b_up, b_down)

cd_arr = np.zeros((len(phi_0), sol_n, 4), dtype='complex') # (sol_n, len(phi_0), 4, 1)
ab_arr = np.zeros_like(cd_arr)

# compute cd_arr
for i in range(len(phi_0)):
    for k in range(sol_n):

        ns = null_space(Matr(roots[i, k], i), rcond=10e-4)

        if ns.size:
            cd_arr[i, k] = ns[:,0]
            
# compute ab_ar
for i in range(len(phi_0)):
    for k in range(sol_n):

        ab_arr[i, k] = ab_vect(cd_arr[i, k], roots[i, k], i)

In [ ]:
# normalize ab and cd amplitudes

for i in range(len(phi_0)):
    for k in range(sol_n):
        
        kapp = np.sqrt(1-roots[i, k]**2)

        sum_msq = np.abs(ab_arr[i, k, 0])**2 + np.abs(ab_arr[i, k, 1])**2 +\
                    np.abs(ab_arr[i, k, 2])**2 + np.abs(ab_arr[i, k, 3])**2 +\
                    np.abs(cd_arr[i, k, 0])**2 + np.abs(cd_arr[i, k, 1])**2 + \
                    np.abs(cd_arr[i, k, 2])**2 + np.abs(cd_arr[i, k, 3])**2
        
        
        new_term = roots[i, k] * np.real(np.conj(ab_arr[i, k, 0])*ab_arr[i, k, 2] + \
                                         np.conj(ab_arr[i, k, 1])*ab_arr[i, k, 3] -\
                                         np.conj(cd_arr[i, k, 0])*cd_arr[i, k, 2] - \
                                         np.conj(cd_arr[i, k, 1])*cd_arr[i, k, 3])


        norm_sq = (tau_1 + kapp * L *(1+r_1**2))/(2 * tau_1 * kapp) * sum_msq - \
                    2*r_1*L/tau_1 * new_term

        ab_arr[i, k] /= np.sqrt(norm_sq) 
        cd_arr[i, k] /= np.sqrt(norm_sq)

### Side computation - computing the current from the grpund state and through the amplitudes

In [ ]:
GS_energy = roots[:,0] + roots[:,1] + roots[:,2] + roots[:,3] 

dx = 2*np.pi/len(phi_0)
GS_current = np.gradient(GS_energy, dx)

In [ ]:
plt.plot(phi_0/np.pi, GS_energy, linewidth=1.2)
plt.grid()
plt.xlabel(r'$\varphi$, $\pi$', fontsize=12)
plt.ylabel(r'$H_0$, $\Delta$', fontsize=12)

In [ ]:
I_c_pl0 = np.max(2*GS_current)
I_c_min0 = np.abs(np.min(2*GS_current))

eta_0 = np.abs((I_c_pl0 - I_c_min0)/(I_c_pl0 + I_c_min0))
eta_0*100

In [ ]:
# computing current for each level through the amplitudes

curr_level3 = np.abs(ab_arr[:, 3, 0])**2 + np.abs(ab_arr[:, 3, 1])**2 -\
        np.abs(ab_arr[:, 3, 2])**2 - np.abs(ab_arr[:, 3, 3])**2 

curr_level2 = np.abs(ab_arr[:, 2, 0])**2 + np.abs(ab_arr[:, 2, 1])**2 -\
        np.abs(ab_arr[:, 2, 2])**2 - np.abs(ab_arr[:, 2, 3])**2  

curr_level1 = np.abs(ab_arr[:, 1, 0])**2 + np.abs(ab_arr[:, 1, 1])**2 -\
        np.abs(ab_arr[:, 1, 2])**2 - np.abs(ab_arr[:, 1, 3])**2 

curr_level0 = np.abs(ab_arr[:, 0, 0])**2 + np.abs(ab_arr[:, 0, 1])**2 -\
        np.abs(ab_arr[:, 0, 2])**2 - np.abs(ab_arr[:, 0, 3])**2 

In [ ]:
for i in range(len(phi_0)):
    if math.isnan(curr_level3[i]):
        curr_level3[i] = 0

In [ ]:
plt.plot(phi_0/np.pi, (curr_level0 + curr_level1 + curr_level2 + curr_level3), label='amplitude')
plt.plot(phi_0/np.pi, (GS_current*2), label='GS')


plt.grid()
plt.legend()
plt.xlabel(r'$\varphi$, $\pi$', fontsize=12)
plt.ylabel(r'$ I_A$, $\Delta$', fontsize=12)

End of the side computation.

In [ ]:
def I_mn(E_m, E_n, ab_arr_m, ab_arr_n, cd_arr_m, cd_arr_n):
    
    a_up_m = ab_arr_m[0]     
    a_down_m = ab_arr_m[1]
    b_up_m = ab_arr_m[2]
    b_down_m = ab_arr_m[3]
    
    a_up_n = ab_arr_n[0]
    a_down_n = ab_arr_n[1]
    b_up_n = ab_arr_n[2]
    b_down_n = ab_arr_n[3]
    
    c_up_m = cd_arr_m[0]
    c_down_m = cd_arr_m[1]
    d_up_m = cd_arr_m[2]
    d_down_m = cd_arr_m[3]
    
    c_up_n = cd_arr_n[0]
    c_down_n = cd_arr_n[1]
    d_up_n = cd_arr_n[2]
    d_down_n = cd_arr_n[3]
    
    gam_m = np.arccos(E_m)
    gam_n = np.arccos(E_n)
    
    sum_spin = np.conj(a_up_m)*a_up_n + np.conj(a_down_m)*a_down_n - \
              (np.conj(b_up_m)*b_up_n + np.conj(b_down_m)*b_down_n) + \
               np.conj(c_up_m)*c_up_n + np.conj(c_down_m)*c_down_n - \
              (np.conj(d_up_m)*d_up_n + np.conj(d_down_m)*d_down_n)
    
    I_mn = 1/(np.sin(np.arccos(E_m)) + np.sin(np.arccos(E_n))) * ((E_m - E_n)/2 * np.sin((gam_m - gam_n)/2) +\
            np.sin((gam_m + gam_n)/2)) * sum_spin
    
    
    return I_mn   

In [ ]:
# array for squared modulus of current matrix elements for transitions between AL lamd -> lamd':
# there are 8 levels, from each level can transit to other 8 levels

# I_AL_sq: first index - phase
#          secound index - index of level from which transition happens
#          third index - to which level transition happens

I_AL_sq = np.zeros((len(phi_0), sol_n, sol_n), dtype = 'complex')

for i in range(len(phi_0)):
    for lamd in range(sol_n):
        for lamd_p in range(sol_n):
            I_AL_sq[i, lamd, lamd_p] = \
            np.abs(I_mn(roots[i, lamd], roots[i, lamd_p], \
                 ab_arr[i, lamd], ab_arr[i, lamd_p], cd_arr[i, lamd], cd_arr[i, lamd_p]))**2

In [ ]:
# transition rates bewteen the ABS

G_AL = np.zeros((len(phi_0), sol_n, sol_n), dtype = 'float64')

for i in range(len(phi_0)):
    for nu in range(sol_n):
        for mu in range(sol_n):
            omega_nu_mu = roots[i, nu] - roots[i, mu]

            G_AL[i, nu, mu] = I_AL_sq[i, mu, nu] * \
            spectral_density(omega_nu_mu, lambda_coupl, damping_param, Omega) * \
            (bose_distrib(omega_nu_mu, T_bos) + 1) 

            if math.isnan(G_AL[i, nu, mu]):
                G_AL[i, nu, mu] = 0

In [ ]:
plt.plot(phi_0[70:130]/np.pi, G_AL[70:130, 4, 3]/np.max(G_AL[70:130, 4, 3]))
plt.scatter(phi_0[70:130]/np.pi, G_AL[70:130, 4, 2]/np.max(G_AL[70:130, 4, 2]), s=5)
plt.plot(phi_0[70:130]/np.pi, G_AL[70:130, 4, 1]/np.max(G_AL[70:130, 4, 1]))

plt.plot(phi_0[70:130]/np.pi, G_AL[70:130, 5, 3]/np.max(G_AL[70:130, 5, 3]))
plt.plot(phi_0[70:130]/np.pi, G_AL[70:130, 5, 2]/np.max(G_AL[70:130, 5, 2]))

plt.plot(phi_0[70:130]/np.pi, G_AL[70:130, 6, 3]/np.max(G_AL[70:130, 6, 3]))

In [ ]:
# # Create a dictionary containing the arrays
data_G = {'phi_0': phi_0/np.pi,
             '1100': G_AL[:, 4, 1],
             '1010': G_AL[:, 4, 2],
             '1001': G_AL[:, 4, 3],
             '0110': G_AL[:, 5, 2],
             '0101': G_AL[:, 5, 3],
             '0011': G_AL[:, 6, 3]}

# # Create a DataFrame from the dictionary
output_df = pd.DataFrame(data_G)

# # Save the DataFrame and arrays to a .dat file
output_df.to_csv('G_phi_bx.dat', sep='\t', index=False)

### Continuum states current, for each phi_0

In [ ]:
def find_ab_matr(E, i):
    
    TM_old =  1 / 2 / np.sqrt(tau_1*tau_2) * (TM_cont(E, i) + TM_cont_Nambu(E, i) )
    
    return TM_old

In [ ]:
def inv_Matr(E, i):
     
    inv_Matr_old = (LA.inv( TM_cont(E, i)/np.sqrt(tau_1*tau_2) ) -\
                    LA.inv( TM_cont_Nambu(E, i)/np.sqrt(tau_1*tau_2) )) 

    return inv_Matr_old

In [ ]:
def convert_cd_to_ab(E, i):
    
    Matr_old = TM_cont(E, i) / np.sqrt(tau_1*tau_2)    

    return Matr_old

In [ ]:
def Matr_cont(E, i):
      
    Matr_old = (TM_cont(E, i) - TM_cont_Nambu(E, i)) / np.sqrt(tau_1*tau_2)
    
    return Matr_old

In [ ]:
# find abcd for s = {1, 2, 3, 4}
def find_abcd(epsilon, i):
  
    ab_arr_1_up = np.zeros((4, 1), dtype='complex128')
    ab_arr_1_down = np.zeros((4, 1), dtype='complex128')
    cd_arr_1_up = np.zeros_like(ab_arr_1_up)
    cd_arr_1_down = np.zeros_like(ab_arr_1_up)
    
    ab_arr_2_up = np.zeros_like(ab_arr_1_up)
    ab_arr_2_down = np.zeros_like(ab_arr_1_up)
    cd_arr_2_up = np.zeros_like(ab_arr_1_up)
    cd_arr_2_down = np.zeros_like(ab_arr_1_up)
    
    ab_arr_3_up = np.zeros_like(ab_arr_1_up)
    ab_arr_3_down = np.zeros_like(ab_arr_1_up)
    cd_arr_3_up = np.zeros_like(ab_arr_1_up)
    cd_arr_3_down = np.zeros_like(ab_arr_1_up)
    
    ab_arr_4_up = np.zeros_like(ab_arr_1_up)
    ab_arr_4_down = np.zeros_like(ab_arr_1_up)
    cd_arr_4_up = np.zeros_like(ab_arr_1_up)
    cd_arr_4_down = np.zeros_like(ab_arr_1_up)
    
    
    # note these are the same for all i's - makes sence to calculate them before phi_0 loop
    sinh_g_2 = 2 * np.sqrt(epsilon**2-1) 
    e_gamma = np.abs(epsilon) + np.sqrt(epsilon**2-1)
    e_m_gamma = np.abs(epsilon) - np.sqrt(epsilon**2-1)
    
    
    cd_arr_1_up = LA.inv(Matr_cont(epsilon, i)) @ np.array([[sinh_g_2], [0], [0], [0]], dtype='complex')
    cd_arr_1_down = LA.inv(Matr_cont(epsilon, i)) @ np.array([[0], [sinh_g_2], [0], [0]], dtype='complex')
    cd_arr_2_up = LA.inv(Matr_cont(epsilon, i)) @ np.array([[0], [0], [-sinh_g_2], [0]], dtype='complex')
    cd_arr_2_down = LA.inv(Matr_cont(epsilon, i)) @ np.array([[0], [0], [0], [-sinh_g_2]], dtype='complex')
    ab_arr_3_up = LA.inv(inv_Matr(epsilon, i)) @ np.array([[-sinh_g_2], [0], [0], [0]], dtype='complex')
    ab_arr_3_down = LA.inv(inv_Matr(epsilon, i)) @ np.array([[0], [-sinh_g_2], [0], [0]], dtype='complex')
    ab_arr_4_up = LA.inv(inv_Matr(epsilon, i)) @ np.array([[0], [0], [sinh_g_2], [0]], dtype='complex')
    ab_arr_4_down = LA.inv(inv_Matr(epsilon, i)) @ np.array([[0], [0], [0], [sinh_g_2]], dtype='complex')
     

    ab_arr_1_up = convert_cd_to_ab(epsilon, i) @ cd_arr_1_up - \
                   np.array([[e_gamma], [0], [0], [0]], dtype='complex')
    ab_arr_1_down = convert_cd_to_ab(epsilon, i) @ cd_arr_1_down - \
                   np.array([[0], [e_gamma], [0], [0]], dtype='complex')
    ab_arr_2_up = convert_cd_to_ab(epsilon, i) @ cd_arr_2_up - \
                   np.array([[0], [0], [e_m_gamma], [0]], dtype='complex')
    ab_arr_2_down = convert_cd_to_ab(epsilon, i) @ cd_arr_2_down - \
                   np.array([[0], [0], [0], [e_m_gamma]], dtype='complex')
    cd_arr_3_up = LA.inv(convert_cd_to_ab(epsilon, i)) @ ab_arr_3_up - \
                np.array([[e_m_gamma], [0], [0], [0]], dtype='complex')
    cd_arr_3_down = LA.inv(convert_cd_to_ab(epsilon, i)) @ ab_arr_3_down - \
                np.array([[0], [e_m_gamma], [0], [0]], dtype='complex')
    cd_arr_4_up = LA.inv(convert_cd_to_ab(epsilon, i)) @ ab_arr_4_up - \
                       np.array([[0], [0], [e_gamma], [0]], dtype='complex')
    cd_arr_4_down = LA.inv(convert_cd_to_ab(epsilon, i)) @ ab_arr_4_down - \
                       np.array([[0], [0], [0], [e_gamma]], dtype='complex')
                       

    norm_1_up = np.sqrt(np.abs(ab_arr_1_up[0])**2 + np.abs(ab_arr_1_up[1])**2 +\
             np.abs(ab_arr_1_up[2])**2 + np.abs(ab_arr_1_up[3])**2 + \
             np.abs(cd_arr_1_up[0])**2 + np.abs(cd_arr_1_up[1])**2 + \
             np.abs(cd_arr_1_up[2])**2 + np.abs(cd_arr_1_up[3])**2)

    ab_arr_1_up /= norm_1_up
    cd_arr_1_up /= norm_1_up
                  

    norm_1_down = np.sqrt(np.abs(ab_arr_1_down[0])**2 + np.abs(ab_arr_1_down[1])**2 +\
             np.abs(ab_arr_1_down[2])**2 + np.abs(ab_arr_1_down[3])**2 + \
             np.abs(cd_arr_1_down[0])**2 + np.abs(cd_arr_1_down[1])**2 + \
             np.abs(cd_arr_1_down[2])**2 + np.abs(cd_arr_1_down[3])**2)

    ab_arr_1_down /= norm_1_down
    cd_arr_1_down /= norm_1_down
                            
    
    norm_2_up = np.sqrt(np.abs(ab_arr_2_up[0])**2 + np.abs(ab_arr_2_up[1])**2 +\
             np.abs(ab_arr_2_up[2])**2 + np.abs(ab_arr_2_up[3])**2 + \
             np.abs(cd_arr_2_up[0])**2 + np.abs(cd_arr_2_up[1])**2 + \
             np.abs(cd_arr_2_up[2])**2 + np.abs(cd_arr_2_up[3])**2)

    ab_arr_2_up /= norm_2_up
    cd_arr_2_up /= norm_2_up
    
    
    norm_2_down = np.sqrt(np.abs(ab_arr_2_down[0])**2 + np.abs(ab_arr_2_down[1])**2 +\
             np.abs(ab_arr_2_down[2])**2 + np.abs(ab_arr_2_down[3])**2 + \
             np.abs(cd_arr_2_down[0])**2 + np.abs(cd_arr_2_down[1])**2 + \
             np.abs(cd_arr_2_down[2])**2 + np.abs(cd_arr_2_down[3])**2)

    ab_arr_2_down /= norm_2_down
    cd_arr_2_down /= norm_2_down
    
    norm_3_up = np.sqrt(np.abs(ab_arr_3_up[0])**2 + np.abs(ab_arr_3_up[1])**2 +\
                 np.abs(ab_arr_3_up[2])**2 + np.abs(ab_arr_3_up[3])**2 + \
                 np.abs(cd_arr_3_up[0])**2 + np.abs(cd_arr_3_up[1])**2 + \
                 np.abs(cd_arr_3_up[2])**2 + np.abs(cd_arr_3_up[3])**2)
        
    ab_arr_3_up /= norm_3_up
    cd_arr_3_up /= norm_3_up 
    
 
    
    norm_3_down = np.sqrt(np.abs(ab_arr_3_down[0])**2 + np.abs(ab_arr_3_down[1])**2 +\
                 np.abs(ab_arr_3_down[2])**2 + np.abs(ab_arr_3_down[3])**2 + \
                 np.abs(cd_arr_3_down[0])**2 + np.abs(cd_arr_3_down[1])**2 + \
                 np.abs(cd_arr_3_down[2])**2 + np.abs(cd_arr_3_down[3])**2)
        
    ab_arr_3_down /= norm_3_down
    cd_arr_3_down /= norm_3_down 


    norm_4_up = np.sqrt(np.abs(ab_arr_4_up[0])**2 + np.abs(ab_arr_4_up[1])**2 +\
                 np.abs(ab_arr_4_up[2])**2 + np.abs(ab_arr_4_up[3])**2 + \
                 np.abs(cd_arr_4_up[0])**2 + np.abs(cd_arr_4_up[1])**2 + \
                 np.abs(cd_arr_4_up[2])**2 + np.abs(cd_arr_4_up[3])**2)
        
    ab_arr_4_up /= norm_4_up
    cd_arr_4_up /= norm_4_up
    
    
    norm_4_down = np.sqrt(np.abs(ab_arr_4_down[0])**2 + np.abs(ab_arr_4_down[1])**2 +\
                 np.abs(ab_arr_4_down[2])**2 + np.abs(ab_arr_4_down[3])**2 + \
                 np.abs(cd_arr_4_down[0])**2 + np.abs(cd_arr_4_down[1])**2 + \
                 np.abs(cd_arr_4_down[2])**2 + np.abs(cd_arr_4_down[3])**2)
        
    ab_arr_4_down /= norm_4_down
    cd_arr_4_down /= norm_4_down
    


    return ab_arr_1_up, cd_arr_1_up, ab_arr_2_up, cd_arr_2_up,\
           ab_arr_3_up, cd_arr_3_up, ab_arr_4_up, cd_arr_4_up, \
           ab_arr_1_down, cd_arr_1_down, ab_arr_2_down, cd_arr_2_down,\
           ab_arr_3_down, cd_arr_3_down, ab_arr_4_down, cd_arr_4_down

In [ ]:
# calculates sum over s of the I_{lamb, (eps, s)} * I_{lamb, (eps, s)}^*
def sum_s_current(epsilon, root_number, i):
    
    E_lambd = roots[i, root_number]
    
    ab_arr_1_up, cd_arr_1_up, ab_arr_2_up, cd_arr_2_up,\
    ab_arr_3_up, cd_arr_3_up, ab_arr_4_up, cd_arr_4_up, \
    ab_arr_1_down, cd_arr_1_down, ab_arr_2_down, cd_arr_2_down,\
    ab_arr_3_down, cd_arr_3_down, ab_arr_4_down, cd_arr_4_down = find_abcd(epsilon, i)
    
    # these do not depend on i
    z = (np.arcsinh(np.sqrt(epsilon**2-1)) + 1j * np.arccos(E_lambd)) / 2
    
    w_pl = np.heaviside(epsilon, 0) * np.sinh(z) + np.heaviside(-epsilon, 0) * np.cosh(z)
    w_min = np.heaviside(epsilon, 0) * np.sinh(-z) + np.heaviside(-epsilon, 0) * np.cosh(-z)
    
    W_z_pl = w_pl + (epsilon-E_lambd)/2 * np.conj(w_pl)
    W_z_min = w_min + (epsilon-E_lambd)/2 * np.conj(w_min)
    
    k_eps = np.sign(epsilon) * np.sqrt(epsilon**2-1)
    kappa_l = np.sqrt(1-E_lambd**2)
    
    
    
    I_1_up = 1/(kappa_l-1j*k_eps) * ((np.conj(ab_arr[i, root_number, 0]) * ab_arr_1_up[0] + \
                                      np.conj(ab_arr[i, root_number, 1]) * ab_arr_1_up[1] - \
                                      np.conj(cd_arr[i, root_number, 2]) * cd_arr_1_up[2] - \
                                      np.conj(cd_arr[i, root_number, 3]) * cd_arr_1_up[3]) * W_z_min + \
                                     (np.conj(ab_arr[i, root_number, 2]) * ab_arr_1_up[2] + \
                                      np.conj(ab_arr[i, root_number, 3]) * ab_arr_1_up[3] - \
                                      np.conj(cd_arr[i, root_number, 0]) * cd_arr_1_up[0] - \
                                      np.conj(cd_arr[i, root_number, 1]) * cd_arr_1_up[1]) * W_z_pl) + \
             1/(kappa_l+1j*k_eps) * (np.conj(ab_arr[i, root_number, 0])) * np.conj(W_z_pl)
    
    I_1_down = 1/(kappa_l-1j*k_eps) * ((np.conj(ab_arr[i, root_number, 0]) * ab_arr_1_down[0] + \
                                      np.conj(ab_arr[i, root_number, 1]) * ab_arr_1_down[1] - \
                                      np.conj(cd_arr[i, root_number, 2]) * cd_arr_1_down[2] - \
                                      np.conj(cd_arr[i, root_number, 3]) * cd_arr_1_down[3]) * W_z_min + \
                                     (np.conj(ab_arr[i, root_number, 2]) * ab_arr_1_down[2] + \
                                      np.conj(ab_arr[i, root_number, 3]) * ab_arr_1_down[3] - \
                                      np.conj(cd_arr[i, root_number, 0]) * cd_arr_1_down[0] - \
                                      np.conj(cd_arr[i, root_number, 1]) * cd_arr_1_down[1]) * W_z_pl) + \
             1/(kappa_l+1j*k_eps) * (np.conj(ab_arr[i, root_number, 1])) * np.conj(W_z_pl)
    
    
    
    I_2_up = 1/(kappa_l-1j*k_eps) * ((np.conj(ab_arr[i, root_number, 0]) * ab_arr_2_up[0] + \
                                      np.conj(ab_arr[i, root_number, 1]) * ab_arr_2_up[1] - \
                                      np.conj(cd_arr[i, root_number, 2]) * cd_arr_2_up[2] - \
                                      np.conj(cd_arr[i, root_number, 3]) * cd_arr_2_up[3]) * W_z_min + \
                                     (np.conj(ab_arr[i, root_number, 2]) * ab_arr_2_up[2] + \
                                      np.conj(ab_arr[i, root_number, 3]) * ab_arr_2_up[3] - \
                                      np.conj(cd_arr[i, root_number, 0]) * cd_arr_2_up[0] - \
                                      np.conj(cd_arr[i, root_number, 1]) * cd_arr_2_up[1]) * W_z_pl) + \
             1/(kappa_l+1j*k_eps) * (np.conj(ab_arr[i, root_number, 2])) * np.conj(W_z_min)
    
    I_2_down = 1/(kappa_l-1j*k_eps) * ((np.conj(ab_arr[i, root_number, 0]) * ab_arr_2_down[0] + \
                                      np.conj(ab_arr[i, root_number, 1]) * ab_arr_2_down[1] - \
                                      np.conj(cd_arr[i, root_number, 2]) * cd_arr_2_down[2] - \
                                      np.conj(cd_arr[i, root_number, 3]) * cd_arr_2_down[3]) * W_z_min + \
                                     (np.conj(ab_arr[i, root_number, 2]) * ab_arr_2_down[2] + \
                                      np.conj(ab_arr[i, root_number, 3]) * ab_arr_2_down[3] - \
                                      np.conj(cd_arr[i, root_number, 0]) * cd_arr_2_down[0] - \
                                      np.conj(cd_arr[i, root_number, 1]) * cd_arr_2_down[1]) * W_z_pl) + \
             1/(kappa_l+1j*k_eps) * (np.conj(ab_arr[i, root_number, 3])) * np.conj(W_z_min)
    
    
    I_3_up = 1/(kappa_l-1j*k_eps) * ((np.conj(ab_arr[i, root_number, 0]) * ab_arr_3_up[0] + \
                                      np.conj(ab_arr[i, root_number, 1]) * ab_arr_3_up[1] - \
                                      np.conj(cd_arr[i, root_number, 2]) * cd_arr_3_up[2] - \
                                      np.conj(cd_arr[i, root_number, 3]) * cd_arr_3_up[3]) * W_z_min + \
                                     (np.conj(ab_arr[i, root_number, 2]) * ab_arr_3_up[2] + \
                                      np.conj(ab_arr[i, root_number, 3]) * ab_arr_3_up[3] - \
                                      np.conj(cd_arr[i, root_number, 0]) * cd_arr_3_up[0] - \
                                      np.conj(cd_arr[i, root_number, 1]) * cd_arr_3_up[1]) * W_z_pl) - \
             1/(kappa_l+1j*k_eps) * (np.conj(cd_arr[i, root_number, 0])) * np.conj(W_z_min)
    
    I_3_down = 1/(kappa_l-1j*k_eps) * ((np.conj(ab_arr[i, root_number, 0]) * ab_arr_3_down[0] + \
                                      np.conj(ab_arr[i, root_number, 1]) * ab_arr_3_down[1] - \
                                      np.conj(cd_arr[i, root_number, 2]) * cd_arr_3_down[2] - \
                                      np.conj(cd_arr[i, root_number, 3]) * cd_arr_3_down[3]) * W_z_min + \
                                     (np.conj(ab_arr[i, root_number, 2]) * ab_arr_3_down[2] + \
                                      np.conj(ab_arr[i, root_number, 3]) * ab_arr_3_down[3] - \
                                      np.conj(cd_arr[i, root_number, 0]) * cd_arr_3_down[0] - \
                                      np.conj(cd_arr[i, root_number, 1]) * cd_arr_3_down[1]) * W_z_pl) - \
             1/(kappa_l+1j*k_eps) * (np.conj(cd_arr[i, root_number, 1])) * np.conj(W_z_min)
    
    
    
    I_4_up = 1/(kappa_l-1j*k_eps) * ((np.conj(ab_arr[i, root_number, 0]) * ab_arr_4_up[0] + \
                                      np.conj(ab_arr[i, root_number, 1]) * ab_arr_4_up[1] - \
                                      np.conj(cd_arr[i, root_number, 2]) * cd_arr_4_up[2] - \
                                      np.conj(cd_arr[i, root_number, 3]) * cd_arr_4_up[3]) * W_z_min + \
                                     (np.conj(ab_arr[i, root_number, 2]) * ab_arr_4_up[2] + \
                                      np.conj(ab_arr[i, root_number, 3]) * ab_arr_4_up[3] - \
                                      np.conj(cd_arr[i, root_number, 0]) * cd_arr_4_up[0] - \
                                      np.conj(cd_arr[i, root_number, 1]) * cd_arr_4_up[1]) * W_z_pl) - \
             1/(kappa_l+1j*k_eps) * (np.conj(cd_arr[i, root_number, 2])) * np.conj(W_z_pl)
    
    I_4_down = 1/(kappa_l-1j*k_eps) * ((np.conj(ab_arr[i, root_number, 0]) * ab_arr_4_down[0] + \
                                      np.conj(ab_arr[i, root_number, 1]) * ab_arr_4_down[1] - \
                                      np.conj(cd_arr[i, root_number, 2]) * cd_arr_4_down[2] - \
                                      np.conj(cd_arr[i, root_number, 3]) * cd_arr_4_down[3]) * W_z_min + \
                                     (np.conj(ab_arr[i, root_number, 2]) * ab_arr_4_down[2] + \
                                      np.conj(ab_arr[i, root_number, 3]) * ab_arr_4_down[3] - \
                                      np.conj(cd_arr[i, root_number, 0]) * cd_arr_4_down[0] - \
                                      np.conj(cd_arr[i, root_number, 1]) * cd_arr_4_down[1]) * W_z_pl) - \
             1/(kappa_l+1j*k_eps) * (np.conj(cd_arr[i, root_number, 3])) * np.conj(W_z_pl)
    
    

    sum_s = 1/np.abs(epsilon) * \
            (I_1_up * np.conj(I_1_up) + I_2_up * np.conj(I_2_up) + \
            I_3_up * np.conj(I_3_up) + I_4_up * np.conj(I_4_up) + \
            I_1_down * np.conj(I_1_down) + I_2_down * np.conj(I_2_down) + \
            I_3_down * np.conj(I_3_down) + I_4_down * np.conj(I_4_down))
           
    
    return sum_s[0]   

In [ ]:
current = sum_s_current(2, 0, 46) 
current_1 = sum_s_current(-2, 4, 46) 

print(current, current_1)